# 01 Data Inventory and Preprocessing

This notebook guides you through checking raw datasets and preparing them for analysis. Preprocessing clips datasets to Lokoja, cleans geometries, and reprojects outputs to the project CRS.

## What you will do

1. List available raw datasets.
2. Confirm the required boundary file exists.
3. Run the preprocessing workflow.
4. Inspect cleaned vectors and reprojected rasters.
5. Record missing datasets for later stages.

## Step 1: Load configuration and helper functions

In [ ]:
from pathlib import Path
from floodsense.config import load_config
from floodsense.paths import list_input_files, first_existing_file

config = load_config()
paths = config['paths']
vector_ext = ['.gpkg', '.geojson', '.json', '.shp']
raster_ext = ['.tif', '.tiff']

## Step 2: Build a simple data inventory

This table helps you see what data is present before analysis. A missing boundary should stop the workflow. Missing optional datasets can be added later.

In [ ]:
import pandas as pd

inventory = []
for key, folder in paths.items():
    if key.startswith('raw_'):
        files = list_input_files(folder)
        inventory.append({
            'dataset': key.replace('raw_', ''),
            'folder': folder,
            'file_count': len(files),
            'example_file': files[0].name if files else None,
        })

inventory_df = pd.DataFrame(inventory)
inventory_df

## Step 3: Confirm the boundary file

The boundary is required because all rasters and vectors need to be clipped to Lokoja. Put a boundary file in `data/raw/boundary/`. Supported vector formats include GPKG, GeoJSON, and Shapefile.

In [ ]:
boundary_file = first_existing_file(paths['raw_boundary'], vector_ext)
if boundary_file is None:
    print('Boundary missing. Add Lokoja boundary data to data/raw/boundary/ before preprocessing.')
else:
    print('Boundary found:', boundary_file)

## Step 4: Run preprocessing

This workflow will:

- clean and dissolve the boundary;
- reproject it to the target CRS;
- save `lokoja_boundary.gpkg` and dashboard GeoJSON;
- process available buildings, roads, and optional rivers;
- clip/reproject available DEM, rainfall, landcover, and population rasters.

If optional datasets are missing, the workflow prints a warning and continues.

In [ ]:
from floodsense.preprocessing import run_preprocessing_workflow

if boundary_file is not None:
    run_preprocessing_workflow(config)
else:
    print('Skipped preprocessing because the boundary is missing.')

## Step 5: Inspect preprocessing outputs

In [ ]:
for label, folder in {
    'cleaned vectors': paths['interim_cleaned'],
    'clipped rasters': paths['interim_clipped'],
    'reprojected rasters': paths['interim_reprojected'],
    'processed vectors': paths['processed_vectors'],
    'dashboard layers': paths['dashboard_layers'],
}.items():
    files = list_input_files(folder)
    print(f'\n{label}: {len(files)} file(s)')
    for file in files[:10]:
        print('  -', file.name)

## What to check before moving on

- `data/processed/vectors/lokoja_boundary.gpkg` exists.
- `data/interim/reprojected/dem.tif` exists before hydrology.
- `data/interim/cleaned/roads.gpkg` and `buildings.gpkg` exist before exposure analysis.

Next notebook: `02_dem_terrain_hydrology.ipynb`.